In [ ]:
import requests
import time
import sys
#from src.utils import logger
from src.data_processing.data_loader import MovieLensDataLoader
from concurrent.futures import ThreadPoolExecutor, as_completed
API_KEY = ""
BASE_URL = "https://api.themoviedb.org/3"
br = 0
def api_get(movieid):
    try:

        url = f"{BASE_URL}/movie/{movieid}?api_key={API_KEY}"
        response = requests.get(url)
        if(response.status_code == 429):
            retry_after = int(response.headers.get("Retry-After", 1))
            #logger.warning(f"Rate limit exceeded. Retrying after {retry_after} seconds.")
            time.sleep(retry_after)
            return api_get(movieid) 
        if response.status_code != 200:
            return None
        credits_url = f"{BASE_URL}/movie/{movieid}/credits?api_key={API_KEY}"
        credits_response = requests.get(credits_url, timeout=5)
        
        cast, director = [], []
        if credits_response.status_code == 200:
            credits_data = credits_response.json()
            cast = [c['name'] for c in credits_data.get('cast', [])[:5]]
            director = [c['name'] for c in credits_data.get('crew', []) if c['job'] == 'Director']
        #print(data.keys())
        data = response.json()
        year = data.get('release_date', '').split('-')[0] if data.get('release_date') else None
        #br += 1
        title = data.get('title', '')
        runtime = data.get('runtime', 0)
        #print(br)
        return        {
            "title": title,
            "year": year,
            "runtime": runtime,
            "cast": cast,
            "director": director
        }
    except Exception as e:
        #logger.error(f"Error fetching data for movie ID {movieid}: {e}")
        return None

def fetch_movie_data(movie_ids):
    br = 0
    movie_data = []
    with ThreadPoolExecutor(max_workers=50) as executor:
        future_to_movieid = {executor.submit(api_get, movieid): movieid for movieid in movie_ids}
        
        for future in as_completed(future_to_movieid):

            movieid = future_to_movieid[future]
            try:
                data = future.result()
                if data:
                    movie_data.append(data)
            except Exception as e:
                #logger.error(f"Error fetching data for movie ID {movieid}: {e}")
                pass
        
    return movie_data
md = MovieLensDataLoader("/home/pajalone/film-recommendation/data/raw/movielens/ml-latest-small/")
md.load_data()
#print(fetch_movie_data([550]))
print(fetch_movie_data(md.links_df['tmdbId'].dropna().astype(int).unique().tolist()))

INFO:src.data_processing.data_loader:Loading MovieLens dataset...
INFO:src.data_processing.data_loader:Loaded 9742 movies
INFO:src.data_processing.data_loader:Loaded 100836 ratings
INFO:src.data_processing.data_loader:Loaded 3683 tags
INFO:src.data_processing.data_loader:Loaded 9742 links


[{'title': 'Grumpier Old Men', 'year': '1995', 'runtime': 101, 'cast': ['Walter Matthau', 'Jack Lemmon', 'Ann-Margret', 'Sophia Loren', 'Daryl Hannah'], 'director': ['Howard Deutch']}, {'title': 'Dracula: Dead and Loving It', 'year': '1995', 'runtime': 88, 'cast': ['Leslie Nielsen', 'Mel Brooks', 'Amy Yasbeck', 'Peter MacNicol', 'Lysette Anthony'], 'director': ['Mel Brooks']}, {'title': 'Father of the Bride Part II', 'year': '1995', 'runtime': 106, 'cast': ['Steve Martin', 'Diane Keaton', 'Martin Short', 'Kimberly Williams-Paisley', 'George Newbern'], 'director': ['Charles Shyer']}, {'title': 'Othello', 'year': '1995', 'runtime': 123, 'cast': ['Laurence Fishburne', 'Irène Jacob', 'Kenneth Branagh', 'Nathaniel Parker', 'Michael Maloney'], 'director': ['Oliver Parker']}, {'title': 'Shanghai Triad', 'year': '1995', 'runtime': 108, 'cast': ['Gong Li', 'Li Baotian', 'Sun Chun', 'Li Xuejian', 'Liu Jiang'], 'director': ['Zhang Yimou']}, {'title': 'Cutthroat Island', 'year': '1995', 'runtime':